# **P1: Framing a Data Problem**
---



## **1. The Question**

**What characteristics of shelter animals are associated with faster adoption?**

Understanding which characteristics are associated with faster adoption could help animal shelters identify factors that may affect how quickly animals find homes. Finding an answer could help shelters make more informed decisions about adoption strategies, such as how animals are presented to potential adopters or which animals may need additional attention to improve their chances of being adopted quickly. As many animals constantly fill up shelters, the space for the animals is reduced and they also get neglected. Quickening the transition from a shelter to a home would be beneficial to both the animals and the shelter, as they will be able to serve more animals, give remaining animals more attention, and feel less pressured to euthanize them as shelter space runs low.

## **2. Sources & access evidence**

**Source 1: Austin Animal Center Intakes (10/01/2013 to 05/05/2025)**

* **Publisher:** City of Austin, Austin Animal Services
* **URL:** Austin Animal Center Intakes dataset/API; "https://data.austintexas.gov/api/v3/views/wter-evkm/query.json?pageNumber=1&pageSize=10"
* **Coverage & Timespan:** Animal intake records for many different kinds of animals entering Austin Animal Center; October 1, 2013 through May 5, 2025
* **Approximate size:** 173,812 records and 16 columns
* **Access method:** City of Austin Open Data Portal API, powered by Socrata
* **Access evidence:** The API returned HTTP status code 200, and a test request successfully retrieved 10 records. A count query returned 173,812 total records.


**Source 2: Austin Animal Center Outcomes (10/01/2013 to 05/05/2025)**

* **Publisher:** City of Austin, Austin Animal Services
* **URL:** Austin Animal Center Outcomes dataset/API; "https://data.austintexas.gov/api/v3/views/9t4d-g238/query.json?accessType=DOWNLOAD"
* **Coverage & Timespan:** Animal outcome records for many different kinds of animals previously at Austin Animal Center; October 1, 2013 through May 5, 2025
* **Approximate size:** 173,775 records and 16 columns
* **Access method:** City of Austin Open Data Portal API, powered by Socrata
* **Access evidence:** The API returned HTTP status code 200 and successfully retrieved 173,775 records.

The two sources can be joined using the shared animal_id field, allowing intake characteristics to be combined with outcome information to examine factors associated with time to adoption.

Likeliest failure point: The biggest risk is that the two sources may not match perfectly on animal_id. If the join produces substantial unmatched records, I will investigate the Animal ID matching and use the available matched records.

In [7]:
import requests
import pandas as pd

intakes_url = "https://data.austintexas.gov/resource/wter-evkm.json"

response = requests.get(
    intakes_url,
    params={"$limit": 10}
)

print("Status code:", response.status_code)

if response.status_code == 200:
    intakes_data = response.json()
    intakes = pd.DataFrame(intakes_data)

    print("Rows retrieved:", len(intakes))
    print(intakes.head())
else:
    print(response.text[:1000])

Status code: 200
Rows retrieved: 10
  animal_id    name                 datetime                datetime2  \
0   A521520    Nina  2013-10-01T07:51:00.000  2013-10-01T07:51:00.000   
1   A664235     NaN  2013-10-01T08:33:00.000  2013-10-01T08:33:00.000   
2   A664236     NaN  2013-10-01T08:33:00.000  2013-10-01T08:33:00.000   
3   A664237     NaN  2013-10-01T08:33:00.000  2013-10-01T08:33:00.000   
4   A664233  Stevie  2013-10-01T08:53:00.000  2013-10-01T08:53:00.000   

                   found_location intake_type intake_condition animal_type  \
0         Norht Ec in Austin (TX)       Stray           Normal         Dog   
1             Abia in Austin (TX)       Stray           Normal         Cat   
2             Abia in Austin (TX)       Stray           Normal         Cat   
3             Abia in Austin (TX)       Stray           Normal         Cat   
4  7405 Springtime in Austin (TX)       Stray          Injured         Dog   

  sex_upon_intake age_upon_intake                       

In [8]:
print("Rows retrieved:", len(intakes))
print("Columns:", len(intakes.columns))
print("Shape:", intakes.shape)

Rows retrieved: 10
Columns: 12
Shape: (10, 12)


In [9]:
count_url = "https://data.austintexas.gov/resource/wter-evkm.json?$select=count(*)"

count_response = requests.get(count_url)

print("Status code:", count_response.status_code)
print(count_response.json())

Status code: 200
[{'count': '173812'}]


In [10]:
import requests
import pandas as pd

outcomes_url = "https://data.austintexas.gov/api/v3/views/9t4d-g238/query.json?accessType=DOWNLOAD"

response = requests.get(outcomes_url)

print("Status code:", response.status_code)

outcomes_data = response.json()
outcomes = pd.DataFrame(outcomes_data)

print("Rows retrieved:", len(outcomes))
print(outcomes.head())

Status code: 200
Rows retrieved: 173775
                  :id           :version               :created_at  \
0  row-w2s2~xrvd_dedm  rv-sudf.bges.qz2y  2025-05-19T14:31:38.129Z   
1  row-8hdn.8mab.73uq  rv-43i2.hxsf~n2rn  2025-05-19T14:31:38.129Z   
2  row-sjd7_t4cn-shy3  rv-u56p~pmk7-ivr3  2025-05-19T14:31:38.129Z   
3  row-n7j6_gyun.f223  rv-ch6a.rn9h.88f7  2025-05-19T14:31:38.129Z   
4  row-7bem-ryu3.e7xz  rv-69gn.ecnj.mmqb  2025-05-19T14:31:38.129Z   

                :updated_at animal_id date_of_birth  \
0  2025-05-19T14:31:38.129Z   A668305    2012-12-01   
1  2025-05-19T14:31:38.129Z   A673335    2012-02-22   
2  2025-05-19T14:31:38.129Z   A675999    2013-04-03   
3  2025-05-19T14:31:38.129Z   A679066    2014-04-16   
4  2025-05-19T14:31:38.129Z   A680855    2014-05-25   

                    datetime monthyear outcome_type outcome_subtype  \
0  2013-12-02T00:00:00-05:00   12-2013     Transfer         Partner   
1  2014-02-22T00:00:00-05:00   02-2014   Euthanasia       Sufferin

In [11]:
print("Rows:", len(outcomes))
print("Columns:", len(outcomes.columns))
print("Shape:", outcomes.shape)

Rows: 173775
Columns: 16
Shape: (173775, 16)


## **3. Constraints & guardrails**

**Source 1: Austin Animal Center Intakes**

Because this source is accessed through the City of Austin Open Data Portal's documented API rather than being scraped, the applicable constraint is the portal's licensing and attribution requirement. The dataset is listed as Public Domain, and the City of Austin Open Data Terms of Use state:


> “Data available through the City of Austin Open Data Portal are offered free and without restriction.”


*The Terms of Use states:*

> “Please provide attribution to both the City of Austin and the City Department that is the source of the cited data.”

I will therefore credit the City of Austin and Austin Animal Services when using this dataset.

*The Terms of Use also states:*

> “You hereby understand and agree that you will not take any measures to re-identify the data or to contact individuals whose information is represented by the data.”

*They further state:*

> “You shall not combine the data with data from any other datasets for the purpose of re-identification of individuals.”

These restrictions do not interfere with the scope of this project because the project focuses on characteristics and outcomes of shelter animals rather than identifying or contacting individuals and the join is done by animal rather than through the use of an individual's information. However, I will remain mindful of this restriction and will not attempt to re-identify or contact any individuals represented in the data.

**Source 2: Austin Animal Center Outcomes**

This source is also accessed through the City of Austin Open Data Portal's documented API. The dataset is listed as Public Domain, and the same Open Data Terms of Use apply. The relevant constraint states:

> “Data available through the City of Austin Open Data Portal are offered free and without restriction.”

*The Terms of Use require:*

> “Please provide attribution to both the City of Austin and the City Department that is the source of the cited data.”

I will therefore credit the City of Austin and Austin Animal Services when using this dataset.

*The Terms of Use states:*

> “You hereby understand and agree that you will not take any measures to re-identify the data or to contact individuals whose information is represented by the data.”

*They further state:*

> “You shall not combine the data with data from any other datasets for the purpose of re-identification of individuals.”


These restrictions do not interfere with the scope of this project because the project focuses on characteristics and outcomes of shelter animals rather than identifying or contacting individuals and the join is done by animal rather than through the use of an individual's information. However, I will remain mindful of this restriction and will not attempt to re-identify or contact any individuals represented in the data.

**Course Guardrails Certification**

Both sources satisfy the course guardrails because they are publicly available through a documented API and are listed as public-domain data. Neither source requires a login or paywall. I will not use personal or identifiable information about individuals. I will also rate-limit API requests, identify requests honestly, and cache the downloaded data rather than repeatedly requesting the same data.

## **4. Tier declaration**

**Tier 1 — Solid.** This project uses two publicly available sources accessed through documented APIs and joins them using the shared animal_id field, before analyzing and cleaning the data, making it an appropriate and manageable level of technical ambition for me, my Python skills and this project. Adding another source to increase the complexity of the scraping and wrangling tasks required would bump me up to **Tier 2 — Ambitious.**

## **Self-Assessment**
**Criteria	Self-Score**
1. Question & motivation	| `4` — Excellent
2. Sources & access evidence | `4` — Excellent
3. Applicable constraints & guardrails |`4` — Excellent
4. Tier declaration & fit |	`4` — Excellent

**Criteria 4 — Tier declaration & fit:** Another week of work could allow me to determine whether adding a more technically challenging component, such as a third source or more complex data extraction, would be appropriate for the project and my skill level and also allow me to determine what source or component specifically would be beneficial to add to the project scope. This would put me in **Tier 2 — Ambitious.**

So even though I'm currently rating myself as Tier 1 (Solid) vs Tier 2 (Ambitious), I did meet all the requirments for this section and included all that was required correctly, completely, and thoughtfully.

**AI Disclosure**

I used ChatGPT to help brainstorm ideas, understand the assignment requirements, check my reasoning, and review some of my wording. I understood and reviewed the information it provided, including independently verifying important information and data access, and made sure I could explain and defend the decisions in this milestone myself.